In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content')

!rm -rf /content/graduation-p
!git clone https://github.com/berk0vic/graduation-p.git /content/graduation-p

os.chdir('/content/graduation-p/notebooks')
sys.path.insert(0, '/content/graduation-p')

!rm -rf /content/graduation-p/data
!ln -s /content/drive/MyDrive/graduation-p/data /content/graduation-p/data

import torch
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html
!pip install -q xgboost pyyaml

In [7]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

Device: cuda


## 1. Load Data

In [ ]:
from src.data.elliptic_loader import build_pyg_graph

graph = build_pyg_graph()

print(f'Nodes: {graph.x.shape[0]:,}')
print(f'Features: {graph.x.shape[1]}')
print(f'Edges: {graph.edge_index.shape[1]:,}')
print()

# Label distribution
labeled_mask = graph.y >= 0  # -1 = unknown, excluded from training
y_labeled = graph.y[labeled_mask]
n_illicit = (y_labeled == 1).sum().item()
n_licit = (y_labeled == 0).sum().item()
n_unknown = (graph.y == -1).sum().item()

print(f'Labeled: {len(y_labeled):,} (illicit: {n_illicit:,}, licit: {n_licit:,})')
print(f'Unknown: {n_unknown:,}')
print(f'Fraud ratio (labeled): {100*n_illicit/len(y_labeled):.1f}%')

## 2. Train/Test Split

In [9]:
# Labeled node'ları train / val / test'e böl (70 / 15 / 15)
from sklearn.model_selection import train_test_split

labeled_indices = labeled_mask.nonzero(as_tuple=True)[0].numpy()
y_labeled_np = y_labeled.numpy()

# First split: 70% train, 30% temp
train_idx, temp_idx = train_test_split(
    labeled_indices, test_size=0.30, stratify=y_labeled_np, random_state=42
)
y_temp = graph.y[temp_idx].numpy()

# Second split: 50/50 of temp → val and test (each 15%)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y_temp, random_state=42
)

train_mask = torch.zeros(graph.x.shape[0], dtype=torch.bool)
val_mask   = torch.zeros(graph.x.shape[0], dtype=torch.bool)
test_mask  = torch.zeros(graph.x.shape[0], dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

print(f'Train: {train_mask.sum().item():,} (fraud: {graph.y[train_mask].sum().item():,})')
print(f'Val:   {val_mask.sum().item():,}   (fraud: {graph.y[val_mask].sum().item():,})')
print(f'Test:  {test_mask.sum().item():,}  (fraud: {graph.y[test_mask].sum().item():,})')


Train: 32,594 (fraud: 3,181)
Val:   6,985   (fraud: 682)
Test:  6,985  (fraud: 682)


## 3. GCN Baseline

In [10]:
from src.models.baselines import GCNBaseline
from src.training.losses import focal_loss
from src.evaluation.metrics import compute_metrics, print_report

gcn = GCNBaseline(in_channels=graph.x.shape[1], hidden=64, out=32).to(device)
optimizer = torch.optim.Adam(gcn.parameters(), lr=0.001, weight_decay=1e-4)

x_dev = graph.x.to(device)
edge_dev = graph.edge_index.to(device)
y_dev = graph.y.float().to(device)

best_state = None
best_loss = float('inf')

for epoch in range(1, 101):
    gcn.train()
    optimizer.zero_grad()
    logits = gcn(x_dev, edge_dev)
    loss = focal_loss(logits[train_mask.to(device)], y_dev[train_mask.to(device)])
    loss.backward()
    optimizer.step()

    if loss.item() < best_loss:
        best_loss = loss.item()
        best_state = {k: v.cpu().clone() for k, v in gcn.state_dict().items()}

    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d} | loss: {loss.item():.4f}')

gcn.load_state_dict(best_state)

# Test
gcn.eval()
with torch.no_grad():
    scores = torch.sigmoid(gcn(x_dev, edge_dev)).cpu().numpy()

test_y = graph.y[test_mask].numpy()
test_scores = scores[test_mask.numpy()]

print('\n=== GCN Baseline (Elliptic) ===')
gcn_metrics = compute_metrics(test_y, test_scores)
print_report(test_y, test_scores)

Epoch  20 | loss: 0.0309
Epoch  40 | loss: 0.0273
Epoch  60 | loss: 0.0256
Epoch  80 | loss: 0.0242
Epoch 100 | loss: 0.0230

=== GCN Baseline (Elliptic) ===

--- Fraud Detection Metrics (threshold=0.512) ---
  threshold           : 0.5123
  f1_fraud            : 0.6180
  recall_fraud        : 0.6833
  precision_fraud     : 0.5642
  roc_auc             : 0.9242
  avg_precision       : 0.6126

  Confusion Matrix:
[[5943  360]
 [ 216  466]]


## 3b. XGBoost Baseline (tabular, no graph)

In [11]:
from src.models.baselines import get_xgboost
from src.evaluation.metrics import compute_metrics, print_report

# Raw features (no graph structure)
X = graph.x.numpy()
y_all = graph.y.numpy()

X_train = X[train_mask.numpy()]
y_train = y_all[train_mask.numpy()]
X_test  = X[test_mask.numpy()]
y_test  = y_all[test_mask.numpy()]

n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
spw = n_neg / n_pos
print(f'scale_pos_weight: {spw:.1f}')

xgb = get_xgboost(scale_pos_weight=spw)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

xgb_scores = xgb.predict_proba(X_test)[:, 1]
print('\n=== XGBoost Baseline (Elliptic) ===')
xgb_metrics = compute_metrics(y_test, xgb_scores)
print_report(y_test, xgb_scores)


scale_pos_weight: 9.2
[0]	validation_0-aucpr:0.92136


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:10:26] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[100]	validation_0-aucpr:0.97535
[200]	validation_0-aucpr:0.98137
[300]	validation_0-aucpr:0.98466
[400]	validation_0-aucpr:0.98594
[499]	validation_0-aucpr:0.98718

=== XGBoost Baseline (Elliptic) ===

--- Fraud Detection Metrics (threshold=0.409) ---
  threshold           : 0.4089
  f1_fraud            : 0.9642
  recall_fraud        : 0.9487
  precision_fraud     : 0.9803
  roc_auc             : 0.9972
  avg_precision       : 0.9872

  Confusion Matrix:
[[6290   13]
 [  35  647]]


## 4. Hybrid GAT+VAE (Elliptic)

Elliptic is a homogeneous graph. We convert it to a single-node-type HeteroData for the Hybrid model.

In [12]:
from torch_geometric.data import HeteroData
from src.models.hybrid_model import HybridGATVAE
import yaml

# Homojen graph'ı tek-tipli hetero graph'a çevir
hetero = HeteroData()
hetero['transaction'].x = graph.x
hetero['transaction'].y = graph.y
hetero['transaction'].train_mask = train_mask
hetero['transaction'].val_mask   = val_mask
hetero['transaction'].test_mask  = test_mask
hetero['transaction', 'linked_to', 'transaction'].edge_index = graph.edge_index

print('HeteroData:')
print(f'  Node types: {hetero.node_types}')
print(f'  Edge types: {hetero.edge_types}')
print(f'  Transaction features: {hetero["transaction"].x.shape}')


HeteroData:
  Node types: ['transaction']
  Edge types: [('transaction', 'linked_to', 'transaction')]
  Transaction features: torch.Size([203769, 166])


In [ ]:
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

in_channels = {'transaction': graph.x.shape[1]}

hybrid_model = HybridGATVAE(
    metadata=hetero.metadata(),
    in_channels=in_channels,
    raw_txn_dim=graph.x.shape[1],
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
)

print(f'Total parameters: {sum(p.numel() for p in hybrid_model.parameters()):,}')

In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(
    model=hybrid_model,
    lr=cfg['training']['lr'],
    weight_decay=cfg['training']['weight_decay'],
    lambda1=cfg['training']['lambda1'],
    lambda2=cfg['training']['lambda2'],
    focal_alpha=cfg['training']['focal_alpha'],
    focal_gamma=cfg['training']['focal_gamma'],
    device=str(device),
)

print('Training started... (100 epochs)')
history = trainer.fit(hetero, val_data=hetero, epochs=100)
print('Training complete!')

In [ ]:
# Hybrid model test results
eval_result = trainer.evaluate(hetero)
hybrid_scores = eval_result['fraud_scores'].numpy()

hybrid_test_scores = hybrid_scores[test_mask.numpy()]

print('=== Hybrid GAT+VAE (Elliptic) ===')
hybrid_metrics = compute_metrics(test_y, hybrid_test_scores)
print_report(test_y, hybrid_test_scores)

## 5. Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['XGBoost (tabular)', 'GCN Baseline', 'Hybrid GAT+VAE'],
    'F1 (Fraud)':        [xgb_metrics['f1_fraud'],     gcn_metrics['f1_fraud'],     hybrid_metrics['f1_fraud']],
    'Recall (Fraud)':    [xgb_metrics['recall_fraud'], gcn_metrics['recall_fraud'], hybrid_metrics['recall_fraud']],
    'Precision (Fraud)': [xgb_metrics['precision_fraud'], gcn_metrics['precision_fraud'], hybrid_metrics['precision_fraud']],
    'AUC-ROC':           [xgb_metrics['roc_auc'],      gcn_metrics['roc_auc'],      hybrid_metrics['roc_auc']],
})

print(results.to_string(index=False))
os.makedirs('../results/tables', exist_ok=True)
results.to_csv('../results/tables/elliptic_results.csv', index=False)
print('\nSaved: results/tables/elliptic_results.csv')

# Bar chart
metrics_to_plot = ['AUC-ROC', 'F1 (Fraud)', 'Recall (Fraud)', 'Precision (Fraud)']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for idx, (_, row) in enumerate(results.iterrows()):
    vals = [row[m] for m in metrics_to_plot]
    ax.bar(x + idx*width, vals, width, label=row['Model'])

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1)
ax.set_title('Elliptic Bitcoin — Model Comparison')
ax.legend()
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/elliptic_comparison.png', dpi=150)
plt.show()

## 6. Detailed Visualizations

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from src.evaluation.metrics import optimal_threshold

gcn_y  = graph.y[test_mask].numpy()
gcn_sc = test_scores
xgb_sc = xgb_scores
hyb_sc = hybrid_test_scores

gcn_pred = (gcn_sc >= optimal_threshold(gcn_y, gcn_sc)).astype(int)
hyb_pred = (hyb_sc >= optimal_threshold(gcn_y, hyb_sc)).astype(int)

# ── 1. ROC Curves + Confusion Matrices ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ax = axes[0]
for label, sc, color in [
    ('XGBoost',        xgb_sc, 'steelblue'),
    ('GCN',            gcn_sc, 'orange'),
    ('Hybrid GAT+VAE', hyb_sc, 'green'),
]:
    fpr, tpr, _ = roc_curve(gcn_y, sc)
    auc = roc_auc_score(gcn_y, sc)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color, lw=2)
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Elliptic'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

for ax, preds, title in [
    (axes[1], gcn_pred, 'GCN Baseline'),
    (axes[2], hyb_pred, 'Hybrid GAT+VAE'),
]:
    cm = confusion_matrix(gcn_y, preds)
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Licit','Illicit']); ax.set_yticklabels(['Licit','Illicit'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix\n{title}')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black',
                    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/elliptic_roc_cm.png', dpi=150); plt.show()

# ── 2. Cross-dataset comparison ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ieee_data = {
    'XGBoost':        {'AUC': 0.940, 'F1': 0.636},
    'GCN':            {'AUC': 0.790, 'F1': 0.293},
    'Hybrid GAT+VAE': {'AUC': 0.842, 'F1': 0.415},
}
elliptic_data = {
    'XGBoost':        {'AUC': xgb_metrics['roc_auc'],    'F1': xgb_metrics['f1_fraud']},
    'GCN':            {'AUC': gcn_metrics['roc_auc'],    'F1': gcn_metrics['f1_fraud']},
    'Hybrid GAT+VAE': {'AUC': hybrid_metrics['roc_auc'], 'F1': hybrid_metrics['f1_fraud']},
}
models = ['XGBoost', 'GCN', 'Hybrid GAT+VAE']
colors = ['steelblue', 'orange', 'green']
x = np.arange(3)
for ax, data, title in [
    (axes[0], ieee_data,     'IEEE-CIS (Credit Card)'),
    (axes[1], elliptic_data, 'Elliptic (Bitcoin)'),
]:
    bars1 = ax.bar(x-0.2, [data[m]['AUC'] for m in models], 0.35, label='AUC-ROC', color=colors, alpha=0.85)
    bars2 = ax.bar(x+0.2, [data[m]['F1']  for m in models], 0.35, label='F1',      color=colors, alpha=0.45)
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=12, ha='right')
    ax.set_ylim(0, 1.05); ax.set_title(title, fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7.5)
fig.suptitle('Cross-Dataset Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/cross_dataset_comparison.png', dpi=150); plt.show()

In [ ]:
# Save model
os.makedirs('../results/models', exist_ok=True)
torch.save(hybrid_model.state_dict(), '../results/models/hybrid_gatvae_elliptic.pt')
print('Model saved.')